# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use regression because `target_next_day_clicks` is a numeric count.

I compare three regression models:

- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

These models fit the lane because the relationship between recent clicks, impressions, search position, CTR, and next-day clicks may be nonlinear.

I use MAE as the primary metric because it represents the average absolute error in predicted clicks and is easy to interpret.

The Week-4 7-day-average baseline remains the benchmark. The goal is not to reward model complexity, but to check whether the models provide useful improvement over the existing baseline.

In [13]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn matplotlib

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [14]:
import os
import duckdb
import pandas as pd
import numpy as np


In [15]:
# Get Hugging Face token securely
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add HF_TOKEN to Colab Secrets."
    )

print("HF_TOKEN found.")

HF_TOKEN found.


In [16]:
# Connect DuckDB to Hugging Face

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB connection ready.")

DuckDB connection ready.


In [17]:
MID_PANEL_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

raw_df = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet('{MID_PANEL_PATH}')
    WHERE gsc_data_available IS TRUE
    ORDER BY report_date
    """
).df()

print("Raw W05 data shape:", raw_df.shape)

display(raw_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw W05 data shape: (3611061, 7)


,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,gsc_avg_position,gsc_data_available
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,2026-03-01,0,4,3.500000,True
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,2026-03-01,0,8,5.875000,True
2,client_62f4a7e64f5e0096,content_39d7361b4945d504,2026-03-01,0,12,4.333333,True
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,2026-03-01,0,5,1.400000,True
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,2026-03-01,0,21,3.476190,True


In [18]:
raw_df = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet('{MID_PANEL_PATH}')
    WHERE gsc_data_available IS TRUE
    ORDER BY report_date
    """
).df()

raw_df["report_date"] = pd.to_datetime(raw_df["report_date"])

print("Raw shape:", raw_df.shape)
display(raw_df.head())

Raw shape: (3611061, 7)


,client_hash_id,content_hash_id,report_date,gsc_clicks,gsc_impressions,gsc_avg_position,gsc_data_available
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0,20,3.350000,True
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0,1,0.000000,True
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,1,125,4.928000,True
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0,7,4.000000,True
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0,11,2.272727,True


In [19]:
feature_query = f"""
WITH daily_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position,

        LEAD(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS target_next_day_clicks

    FROM read_parquet('{MID_PANEL_PATH}')

    WHERE gsc_data_available IS TRUE
),

feature_rows AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        AVG(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS clicks_7d_avg,

        AVG(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS impressions_7d_avg,

        AVG(gsc_avg_position) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS position_7d_avg,

        (
            SUM(gsc_clicks) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) * 1.0
            /
            NULLIF(
                SUM(gsc_impressions) OVER (
                    PARTITION BY client_hash_id, content_hash_id
                    ORDER BY report_date
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
                ),
                0
            )
        ) AS ctr_7d,

        CASE
            WHEN DAYOFWEEK(report_date) IN (0, 6)
            THEN 1
            ELSE 0
        END AS is_weekend,

        target_next_day_clicks

    FROM daily_data
)

SELECT *
FROM feature_rows
WHERE target_next_day_clicks IS NOT NULL
ORDER BY report_date
"""

raw_df = con.sql(feature_query).df()

print("Feature dataset shape:", raw_df.shape)

display(raw_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature dataset shape: (3434323, 9)


,client_hash_id,content_hash_id,report_date,clicks_7d_avg,impressions_7d_avg,position_7d_avg,ctr_7d,is_weekend,target_next_day_clicks
0,client_08a6a72ff48e62c0,content_012a0a2d830c7767,2026-03-01,0.0,6.0,0.333333,0.0,1,0
1,client_08a6a72ff48e62c0,content_01a56a1223cc730e,2026-03-01,0.0,8.0,7.750000,0.0,1,0
2,client_08a6a72ff48e62c0,content_030ab35db9d8d273,2026-03-01,0.0,2.0,12.000000,0.0,1,0
3,client_08a6a72ff48e62c0,content_04a40af41071b3cb,2026-03-01,0.0,1.0,0.000000,0.0,1,0
4,client_08a6a72ff48e62c0,content_04c5b8a44d3c980d,2026-03-01,0.0,5.0,8.800000,0.0,1,0


In [20]:
FEATURES = [
    "clicks_7d_avg",
    "impressions_7d_avg",
    "position_7d_avg",
    "ctr_7d",
    "is_weekend"
]

TARGET = "target_next_day_clicks"

print(raw_df.columns.tolist())

missing = [
    col for col in FEATURES + [TARGET]
    if col not in raw_df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("✓ All W05 features are available.")

['client_hash_id', 'content_hash_id', 'report_date', 'clicks_7d_avg', 'impressions_7d_avg', 'position_7d_avg', 'ctr_7d', 'is_weekend', 'target_next_day_clicks']
✓ All W05 features are available.


In [21]:
raw_df["report_date"] = pd.to_datetime(
    raw_df["report_date"]
)

dates = np.sort(
    raw_df["report_date"]
    .dropna()
    .unique()
)

cutoff = dates[
    int(len(dates) * 0.8)
]

train_df = raw_df[
    raw_df["report_date"] < cutoff
].copy()

test_df = raw_df[
    raw_df["report_date"] >= cutoff
].copy()

X_train = train_df[FEATURES].copy()
X_test = test_df[FEATURES].copy()

y_train = train_df[TARGET].astype(float)
y_test = test_df[TARGET].astype(float)

print("Cutoff:", pd.Timestamp(cutoff).date())
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Train period:",
    train_df["report_date"].min().date(),
    "to",
    train_df["report_date"].max().date()
)

print(
    "Test period:",
    test_df["report_date"].min().date(),
    "to",
    test_df["report_date"].max().date()
)

Cutoff: 2026-03-25
Train rows: 2713724
Test rows: 720599
Train period: 2026-03-01 to 2026-03-24
Test period: 2026-03-25 to 2026-03-30


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
models = {
    "Decision Tree": DecisionTreeRegressor(
        max_depth=6,
        min_samples_leaf=30,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=20,
        random_state=42
    )
}

In [23]:
trained_models = {}
predictions = {}
results = []

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    # Clicks cannot be negative
    pred = np.clip(
        pred,
        0,
        None
    )

    trained_models[name] = model
    predictions[name] = pred

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

model_results = pd.DataFrame(results)

display(model_results)

Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...


,Model,MAE,RMSE,R2
0,Decision Tree,0.226638,0.876170,0.613155
1,Random Forest,0.224806,0.866462,0.621680
2,Gradient Boosting,0.225113,0.821733,0.659732


In [26]:
baseline_mae = 0.0 # Placeholder, replace with actual Week-4 baseline MAE
baseline_rmse = 0.0 # Placeholder, replace with actual Week-4 baseline RMSE
baseline_r2 = 0.0 # Placeholder, replace with actual Week-4 baseline R2

comparison = pd.concat(
    [
        pd.DataFrame([
            {
            "Model": "Week-4 Baseline",
            "MAE": baseline_mae,
            "RMSE": baseline_rmse,
            "R2": baseline_r2
        }
        ]),
        model_results
    ],
    ignore_index=True
)

comparison = (
    comparison
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(
    comparison.style.format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "R2": "{:.4f}"
    })
)

,Model,MAE,RMSE,R2
0,Week-4 Baseline,0.0000,0.0000,0.0000
1,Random Forest,0.2248,0.8665,0.6217
2,Gradient Boosting,0.2251,0.8217,0.6597
3,Decision Tree,0.2266,0.8762,0.6132


In [27]:
best_row = (
    comparison
    .sort_values("MAE")
    .iloc[0]
)

print("Best model:", best_row["Model"])
print(f"Best MAE: {best_row['MAE']:.4f}")

Best model: Week-4 Baseline
Best MAE: 0.0000


In [28]:
best_model_name = (
    model_results
    .sort_values("MAE")
    .iloc[0]["Model"]
)

best_mae = (
    model_results
    .sort_values("MAE")
    .iloc[0]["MAE"]
)

mae_improvement = baseline_mae - best_mae

improvement_percent = (
    mae_improvement / baseline_mae
) * 100

print(f"Week-4 baseline MAE : {baseline_mae:.4f}")
print(f"Best W05 model      : {best_model_name}")
print(f"Best W05 MAE        : {best_mae:.4f}")
print(f"MAE improvement     : {mae_improvement:.4f}")
print(f"Improvement         : {improvement_percent:.2f}%")

Week-4 baseline MAE : 0.0000
Best W05 model      : Random Forest
Best W05 MAE        : 0.2248
MAE improvement     : -0.2248
Improvement         : -inf%


/tmp/ipykernel_8006/42644757.py:16: RuntimeWarning: divide by zero encountered in scalar divide
  mae_improvement / baseline_mae


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [30]:
# Create error analysis table
best_pred = predictions[best_model_name]

error_df = test_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        *FEATURES,
        TARGET
    ]
].copy()

error_df["prediction"] = best_pred

error_df["absolute_error"] = (
    error_df[TARGET] - error_df["prediction"]
).abs()

error_df["error"] = (
    error_df[TARGET] - error_df["prediction"]
)

display(
    error_df
    .sort_values("absolute_error", ascending=False)
    .head(10)
)

,report_date,client_hash_id,content_hash_id,clicks_7d_avg,impressions_7d_avg,position_7d_avg,ctr_7d,is_weekend,target_next_day_clicks,prediction,absolute_error,error
3337245,2026-03-30,client_23a62021009f63c4,content_e6df0936699f5b8f,20.000000,717.857143,32.203914,0.027861,0,269,15.832420,253.167580,253.167580
3351003,2026-03-30,client_20259bd6705d81d4,content_fa4b9e9229816684,7.857143,254.000000,9.381161,0.030934,0,202,6.725406,195.274594,195.274594
3381508,2026-03-30,client_23a62021009f63c4,content_74de5f247659e956,5.857143,902.714286,2.165923,0.006488,0,170,5.477088,164.522912,164.522912
2966343,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,211.571429,32126.857143,2.316705,0.006586,0,271,127.244486,143.755514,143.755514
2979321,2026-03-27,client_23a62021009f63c4,content_66288edeb93b7c4f,49.571429,7723.857143,12.067160,0.006418,0,165,34.366774,130.633226,130.633226
3088257,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,222.571429,33542.428571,2.287525,0.006636,1,252,127.244486,124.755514,124.755514
3313558,2026-03-29,client_23a62021009f63c4,content_9f6d1003cf37e2d9,44.857143,2492.714286,9.920424,0.017995,1,138,26.271959,111.728041,111.728041
3324902,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,228.285714,34911.285714,2.233735,0.006539,0,235,126.848223,108.151777,108.151777
3207198,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,227.000000,34491.000000,2.260189,0.006581,1,225,127.244486,97.755514,97.755514
2842648,2026-03-26,client_e547b89c05043229,content_eadb33b5df496f4a,208.000000,30490.857143,2.356220,0.006822,0,223,127.244486,95.755514,95.755514


In [32]:
# Feature importance
best_model = trained_models[best_model_name]

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": best_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance_df)

,feature,importance
0,clicks_7d_avg,0.641957
1,impressions_7d_avg,0.239277
3,ctr_7d,0.101173
2,position_7d_avg,0.017043
4,is_weekend,0.000551


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.